# 04 — Ensemble & Transfer Learning

- Soft voting ensemble: TS+LR (0.4) + EEGNet (0.3) + ShallowConvNet (0.3)
- Cross-patient transfer via Riemannian Procrustes Analysis (RPA)

**Note:** Ensemble requires PyTorch + braindecode (may be slow without GPU).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import mne
from sklearn.model_selection import StratifiedKFold

from src.loading import load_all_patients
from src.classifiers import build_ensemble
from src.transfer import cross_patient_transfer
from src.evaluation import full_evaluation

mne.set_log_level('WARNING')

## 4.1 Load Data

In [ ]:
patient_data = load_all_patients('../data/')

## 4.2 Ensemble Evaluation

Manual 5-fold CV since the ensemble trains internally.

In [ ]:
for pid, (X, y, _) in patient_data.items():
    print(f'\n{"="*50}')
    print(f'Ensemble — {pid}')
    print(f'{"="*50}')
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_accs = []
    
    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        ensemble = build_ensemble(X_train, y_train)
        y_pred = ensemble.predict(X_test)
        acc = np.mean(y_pred == y_test)
        fold_accs.append(acc)
        print(f'  Fold {fold+1}: {acc:.3f}')
    
    print(f'  Mean: {np.mean(fold_accs):.3f} ± {np.std(fold_accs):.3f}')

## 4.3 Cross-Patient Transfer Learning (RPA)

Leave-one-patient-out: train on all others, test on target.
Compare RPA-aligned vs no-transfer baseline.

In [ ]:
transfer_results = cross_patient_transfer(patient_data)
display(transfer_results)

In [ ]:
# Visualize transfer improvement
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(transfer_results))
width = 0.35

ax.bar(x - width/2, transfer_results['no_transfer_accuracy'], width,
       label='No Transfer', color='#EF5350')
ax.bar(x + width/2, transfer_results['rpa_accuracy'], width,
       label='RPA Transfer', color='#42A5F5')

ax.set_xlabel('Target Patient')
ax.set_ylabel('Accuracy')
ax.set_title('Cross-Patient Transfer: RPA vs No Transfer')
ax.set_xticks(x)
ax.set_xticklabels(transfer_results['target_patient'])
ax.axhline(y=0.5, color='gray', linestyle='--', label='Chance')
ax.legend()
plt.tight_layout()
plt.savefig('../results/transfer_comparison.png', dpi=300)
plt.show()